In [1]:
var('n e_t u_bar v_bar f_4t f_5t g_4t g_5t Xi dt du dv gamma F_phi e_phi y K G m')
var('u_bar m G eta gamma e_t n Xi')
var('y y_0 h TE y_1PN y_15PN y_2PN y_25PN Xi chi_A chi_S delta omega')
var('e_t gamma y_inv e0 omega_dash A B flagTail e0 x0 flagHF SO chi w0 f0 j f_var f0_var e y0 kappaA kappaS horiz')

(e_t,
 gamma,
 y_inv,
 e0,
 omega_dash,
 A,
 B,
 flagTail,
 e0,
 x0,
 flagHF,
 SO,
 chi,
 w0,
 f0,
 j,
 f_var,
 f0_var,
 e,
 y0,
 kappaA,
 kappaS,
 horiz)

In [2]:
import re
filename = "henry/de_dt_final.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"de_dt_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    de_dt = ((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand()
    #*(1/(Xi^3*G))



filename = "henry/dx_dt_final.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"dx_dt_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    dx_dt = ((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand()
    #*(Xi^3/G)


filename = "henry/dx_dt_horiz.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"dx_dt_horiz\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    dx_dt_horiz = (((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand())*horiz   


filename = "henry/de_dt_horiz.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"de_dt_horiz\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    de_dt_horiz = (((sage_eval(expr_str, locals=globals())).subs({e:e_t})).expand())*horiz    

In [3]:
import re
filename = "henry/xe.txt"

with open(filename, "r") as f:
    content = f.read()


match = re.search(r"xe_final\s*=\s*(.*)", content, re.DOTALL)
if match:
    expr_str = match.group(1).strip()
    xe_expr = ((sage_eval(expr_str, locals=globals()))*(e^(12/19)/e0^(12/19))).subs({e:e_t}).expand()

In [4]:
def truncate_var(expr,var, order=2):
    """
    Truncate a Sage expression in Xi to the given order (default 3).
    Keeps terms up to Xi^order and discards higher powers.
    """
    truncated = sum(expr.coefficient(var,n) * var^n for n in range(order + 1))
    return truncated

**[Henry,2023] claims that their non spinning dx_dt and de_dt (pg.17) agree with [Michael,2019]**

**Non spinning terms (>2 PN) are missing in Henry files**

[Henry,2023]
https://journals.aps.org/prd/pdf/10.1103/PhysRevD.108.104016

[Michael,2019]
https://arxiv.org/pdf/1906.06263

**[Michael,2019] took non-spinning terms from Arun[2009]**

Arun[2009]
https://arxiv.org/pdf/0908.3854

Tiwari[2019]
https://arxiv.org/pdf/1905.07956

Tanay[2016] https://arxiv.org/pdf/1602.03081

**The problem is the difference in expressions for non spinning dx_dt at 2 PN given in Henry (fluxes.dat.m file) and Michael (Eq. B1 onwards)**

**CHECK IF HENRY AND TIWARI MATCH (UPTO 2 PN)**

**Newtonian**

In [5]:
#e_prefactor=-(304/15)*((eta*e_t*x^4)/(15*m))

In [6]:
e_N_file=((de_dt.coefficient(Xi,0)).expand().taylor(e_t,0,7).factor()/e_t).expand()

In [7]:
show((e_N_file/e_N_file.coefficient(e_t,0)).expand())

20195/2432*e_t^6 + 3265/608*e_t^4 + 881/304*e_t^2 + 1

In [8]:
def get_PN(de_dt, pn):
    e_file = (de_dt.coefficient(Xi, pn).expand().taylor(e_t, 0, 7).factor()/ e_t).expand()

    return (e_file / e_N_file.coefficient(e_t, 0)).expand().coefficient(x,pn/2)

In [9]:
e_1PN_file=get_PN(de_dt, 2)

In [10]:
show(e_1PN_file)

-2501905/14592*e_t^6*eta + 121833/1216*e_t^6 - 298115/3648*e_t^4*eta + 87749/2128*e_t^4 - 51847/1824*e_t^2*eta + 40115/4256*e_t^2 - 1021/228*eta - 2817/2128

In [11]:
e_15PN_file=get_PN(de_dt, 3)

In [12]:
show(e_15PN_file.coefficient(SO,0).coefficient(flagTail).coefficient(pi))

161865935/700416*e_t^6 + 3061465/29184*e_t^4 + 21729/608*e_t^2 + 985/152

In [13]:
e_2PN_file=get_PN(de_dt, 4)

In [14]:
show(e_2PN_file.coefficient(SO,0).coefficient(e_t,6))

87633595/58368*eta^2 - 65394101/58368*eta - 13488023/408576

In [15]:
#2/3 *1/sqrt(x) domega_dt 

In [16]:
x_N_file=(((dx_dt/x^5).expand().coefficient(Xi,0)).taylor(e_t,0,6))*(3/2)

In [17]:
show((x_N_file/x_N_file.coefficient(e_t,0)).expand())

3815/96*e_t^6 + 605/32*e_t^4 + 157/24*e_t^2 + 1

In [18]:
def get_dx_PN(dx_dt, pn):
    x_file = (((dx_dt/x^5).expand().coefficient(Xi,pn)).taylor(e_t,0,6))*(3/2)

    return (x_file / x_N_file.coefficient(e_t, 0)).expand().coefficient(x,pn/2)

In [19]:
x_1PN_file=get_dx_PN(dx_dt, 2)

In [20]:
show(x_1PN_file.coefficient(e_t,6))

-107765/192*eta + 13823/48

In [21]:
x_15PN_file=get_dx_PN(dx_dt, 3)

In [22]:
show(x_15PN_file.coefficient(e_t,0).coefficient(SO,0).coefficient(flagTail).coefficient(pi))

4

In [23]:
x_2PN_file=get_dx_PN(dx_dt, 4)

In [24]:
show(x_2PN_file.coefficient(e_t,0).coefficient(SO,0))

59/18*eta^2 + 13661/2016*eta + 34103/18144

**B1a) - B2d) in https://arxiv.org/pdf/1905.07956 verified**

**Can we use above paper non spinning expressions for >2.5 PN because they are taylor expanded?**

In [25]:
dx_de_N_integrand=((truncate_var(dx_dt,Xi,order=0).factor()/truncate_var(de_dt,Xi,order=0).factor()))/x

In [26]:
dx_de_N=dx_de_N_integrand.taylor(e_t,0,6)

In [27]:
show(dx_de_N)

-273266243/133448704*e_t^5 - 825317/438976*e_t^3 - 3323/1444*e_t - 12/19/e_t

In [28]:
dx_file_N=dx_dt.coefficient(Xi,0).expand().taylor(e_t,0,6).factor()

In [29]:
de_file_N=de_dt.coefficient(Xi,0).expand().taylor(e_t,0,6).factor()

In [30]:
show((dx_file_N/de_file_N).taylor(e_t,0,6))

-973144163/133448704*e_t^5*x - 825317/438976*e_t^3*x - 3323/1444*e_t*x - 12/19*x/e_t

NO YOU CANNOT

**Cross Checking Henry (dx_dt and de_dt) with Michael upto 2 PN**

In [34]:
dx_1PN_taylor=((((dx_dt.coefficient(Xi,2).coefficient(SO,0).numerator())/(dx_dt.coefficient(Xi,2).coefficient(SO,0).denominator())).subs({(-e_t^2+1)^(-9/2):y}).coefficient(y))*((3*m)/(2*eta*x^5))).expand()

In [41]:
dx_2PN_taylor=((dx_dt.coefficient(Xi,4).coefficient(SO,0).subs({(-e_t^2+1)^(-11/2):y}).subs({(e_t^2-1)^(-5):-y/sqrt(-e_t^2+1)}).coefficient(y)*((3*m)/(2*eta*x^5))).expand())

In [48]:
show(dx_2PN_taylor.coefficient(e_t,6).collect(e_t))

192943/90*eta^2*x^2 - 977051/240*eta*x^2 - 35*eta*x^2/sqrt(-e_t^2 + 1) + 4634689/1680*x^2 + 175/2*x^2/sqrt(-e_t^2 + 1)

In [57]:
dx_2PN_taylor_2=(dx_2PN_taylor-dx_2PN_taylor.coefficient((-e_t^2+1)^(-1/2))*(-e_t^2+1)^(-1/2)).expand()

In [ ]:
show(dx_2PN_taylor_2.coefficient(e_t,6))

**Checking 1.5 PN non spinning Henry vs Michael**

In [86]:
show(dx_dt.coefficient(Xi,3).coefficient(SO,0).subs({e_t^10 - 5*e_t^8 + 10*e_t^6 - 10*e_t^4 + 5*e_t^2 - 1:(e_t^2-1)^(5)}).taylor(e_t,0,6))

6204647/720*pi*e_t^6*eta*flagTail*x^(13/2)/m + 8591/3*pi*e_t^4*eta*flagTail*x^(13/2)/m + 1868/3*pi*e_t^2*eta*flagTail*x^(13/2)/m + 256/5*pi*eta*flagTail*x^(13/2)/m

In [87]:
phi_og = 1 + (2335/192)*e_t^2 + (42955/768)*e_t^4 + (6204647/36864)*e_t^6
dx_15_michael=(96/5)*(2*eta/(3*m))*(4*pi*phi_og)

In [88]:
show(dx_15_michael.expand())

6204647/720*pi*e_t^6*eta/m + 8591/3*pi*e_t^4*eta/m + 1868/3*pi*e_t^2*eta/m + 256/5*pi*eta/m

In [57]:
de_15_taylor=(de_dt.coefficient(Xi,3).coefficient(SO,0).subs({e_t^8-4*e_t^6+6*e_t^4-4*e_t^2+1:(-e_t^2+1)^(4)}))

In [69]:
show(de_15_taylor.taylor(e_t,0,6))

-612293/288*pi*e_t^5*eta*flagTail*x^(11/2)/m - 7243/10*pi*e_t^3*eta*flagTail*x^(11/2)/m - 394/3*pi*e_t*eta*flagTail*x^(11/2)/m

In [64]:
phi_og = 1 + (2335/192)*e_t^2 + (42955/768)*e_t^4 + (6204647/36864)*e_t^6

phi_tilde_og = 1 + (209/32)*e_t^2 + (2415/128)*e_t^4 + (730751/18432)*e_t^6

In [65]:
phi_e = (192/985) * sqrt(1 - e_t^2) / e_t^2 * (
    sqrt(1 - e_t^2) * phi_og - phi_tilde_og
)

In [66]:
de_15_michael=(32/5)*(-e_t*eta*x^4/m)*(-985*pi*x^(3/2)/48)*phi_e

In [70]:
show(de_15_michael.taylor(e_t,0,6))

612293/288*pi*e_t^5*eta*x^(11/2)/m + 7243/10*pi*e_t^3*eta*x^(11/2)/m + 394/3*pi*e_t*eta*x^(11/2)/m

**1.5 PN matches**

**dx_dt doesnt match. Please check 2 PN piece**

**Henry vs Tanay (2 PN)**

In [102]:
dx_2PN_taney=((dx_dt.coefficient(Xi,4).coefficient(SO,0).subs({(-e_t^2+1)^(-11/2):y}).subs({(e_t^2-1)^(-5):-y/sqrt(-e_t^2+1)}).coefficient(y)*((m)/(eta*x^5))).expand())

In [103]:
dx_2PN_taney_2=(dx_2PN_taney-dx_2PN_taney.coefficient((-e_t^2+1)^(-1/2))*(-e_t^2+1)^(-1/2)).expand()

In [104]:
show((dx_2PN_taney_2.coefficient(e_t,2)*45360).expand())

61282032*eta^2*x^2 + 15464736*eta*x^2 - 92846560*x^2

In [37]:
show((dx_2PN_taney.coefficient((-e_t^2+1)^(-1/2))*45360).expand().factor())

-3024*(175*e_t^6 + 4386*e_t^4 + 4268*e_t^2 + 96)*(2*eta - 5)*x^2

**Henry matches Tanay Eq. B9b)**

**de_dt : Michael vs Henry vs Tanay**

Michael

In [47]:
show((((de_dt.coefficient(Xi,2).subs({(-e_t^2+1)^(-7/2):y})/y).expand())*((-m)/(e_t*eta*x^4))).expand().coefficient(e_t,4))

-1664/45*eta*x + 13929/280*x

In [48]:
de_2PN_michael=((de_dt.coefficient(Xi,4).coefficient(SO,0).expand().subs({e_t^8-4*e_t^6+6*e_t^4-4*e_t^2+1:(-e_t^2+1)^(4)}).subs({(-e_t^2+1)^(-9/2):y}).subs({(e_t^2-1)^(-4):y/sqrt(-e_t^2+1)}).coefficient(y)*((-m)/(e_t*eta*x^4))).expand())

In [52]:
de_2PN_michael_2=(de_2PN_michael-de_2PN_michael.coefficient((-e_t^2+1)^(-1/2))*(-e_t^2+1)^(-1/2)).expand()

In [54]:
show(de_2PN_michael_2.coefficient(e_t,2))

64433/40*eta^2*x^2 - 988423/840*eta*x^2 - 3082783/2520*x^2

In [42]:
show((de_2PN_michael.coefficient((-e_t^2+1)^(-1/2))).expand())

-113/3*e_t^4*eta*x^2 + 565/6*e_t^4*x^2 - 2321/5*e_t^2*eta*x^2 + 2321/2*e_t^2*x^2 - 2672/15*eta*x^2 + 1336/3*x^2

Taney

In [43]:
de_2PN_taney=((de_dt.coefficient(Xi,4).coefficient(SO,0).expand().subs({e_t^8-4*e_t^6+6*e_t^4-4*e_t^2+1:(-e_t^2+1)^(4)}).subs({(-e_t^2+1)^(-9/2):y}).subs({(e_t^2-1)^(-4):y/sqrt(-e_t^2+1)}).coefficient(y)*((-30240*m)/(e_t*eta*x^4))).expand())

In [44]:
de_2PN_taney_2=(de_2PN_taney-de_2PN_taney.coefficient((-e_t^2+1)^(-1/2))*(-e_t^2+1)^(-1/2)).expand()

In [45]:
show(de_2PN_taney_2.coefficient(e_t,6))

2758560*eta^2*x^2 - 4344852*eta*x^2 + 3786543*x^2

In [46]:
show((de_2PN_taney.coefficient((-e_t^2+1)^(-1/2))).expand().factor())

-1008*(565*e_t^4 + 6963*e_t^2 + 2672)*(2*eta - 5)*x^2

**Henry matches both refs [Michael,Taney] for de_dt**